# Feature-Selection Experiments for Anomaly Detection — Combined Data

This notebook evaluates Isolation Forest, Local Outlier Factor, and One-Class SVM on the full combined representation and four unsupervised feature-selection/reduction methods: Correlation Filtering, Variance Threshold, Laplacian Score, and PCA. All preprocessing, feature selection, and detector fitting use **normal-only training traffic**. Held-out labels are used only for evaluation.

MLflow run names follow `combined_{Featureselection}_{Algorithm}`, for example `combined_variancethreshold_IsolationForest`.

In [1]:
import platform, sys, time
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import mlflow, mlflow.sklearn
import numpy as np
import pandas as pd
import sklearn
from IPython.display import Markdown, display
from mlflow.models import infer_signature
from scipy.sparse import csgraph
from sklearn.base import BaseEstimator
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay, accuracy_score, average_precision_score, balanced_accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor, kneighbors_graph
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.svm import OneClassSVM
PROJECT_ROOT=Path.cwd()
if not (PROJECT_ROOT/'configs').exists(): PROJECT_ROOT=PROJECT_ROOT.parent
sys.path.insert(0,str(PROJECT_ROOT))
from configs.config import EXPERIMENT_NAME, RANDOM_STATE
TRACKING_DB=(PROJECT_ROOT/'Notebooks'/'mlflow.db').resolve(); mlflow.set_tracking_uri(f'sqlite:///{TRACKING_DB.as_posix()}'); mlflow.set_experiment(EXPERIMENT_NAME)
NORMAL_TRAIN_LIMIT=6000; SELECTION_SAMPLE_SIZE=4000
CORRELATION_THRESHOLD=0.90; VARIANCE_THRESHOLD=0.01; PCA_VARIANCE_TO_KEEP=0.95; LAPLACIAN_TOP_K=36; LAPLACIAN_NEIGHBORS=10
ALGORITHMS=['IsolationForest','LocalOutlierFactor','OneClassSVM']
print('MLflow:',mlflow.get_tracking_uri(),'| experiment:',EXPERIMENT_NAME)

MLflow: sqlite:///D:/E Drive/Sentiflow-Network Intrusion Detection/Notebooks/mlflow.db | experiment: Network_Intrusion_Detection


In [2]:
data_path=PROJECT_ROOT/'Data'/'Consolidated_df.csv'; df=pd.read_csv(data_path)
ORIGINAL_FEATURES=('duration','protocoltype','service','flag','srcbytes','dstbytes','land','wrongfragment','urgent','hot','numfailedlogins','loggedin','numcompromised','rootshell','suattempted','numroot','numfilecreations','numshells','numaccessfiles','numoutboundcmds','ishostlogin','isguestlogin','count','srvcount','serrorrate','srvserrorrate','rerrorrate','srvrerrorrate','samesrvrate','diffsrvrate','srvdiffhostrate','dsthostcount','dsthostsrvcount','dsthostsamesrvrate','dsthostdiffsrvrate','dsthostsamesrcportrate','dsthostsrvdiffhostrate','dsthostserrorrate','dsthostsrvserrorrate','dsthostrerrorrate','dsthostsrvrerrorrate')
ENGINEERED_FEATURES=('total_bytes','bytes_per_second','src_dst_byte_ratio','src_byte_fraction','dst_byte_fraction','byte_asymmetry','service_connection_ratio','host_service_ratio','different_service_connections','different_host_service_connections','short_term_scan_pressure','host_scan_pressure','same_source_port_pressure','short_term_serror_score','short_term_rerror_score','short_term_error_score','host_serror_score','host_rerror_score','host_error_score','same_service_rate_gap','different_service_rate_gap','serror_rate_gap','rerror_rate_gap','authentication_risk_score','has_failed_login','failed_login_and_logged_in','suspicious_admin_activity','privileged_activity_score','content_risk_score','file_and_shell_activity','root_compromise_ratio')
COMBINED_FEATURES=ORIGINAL_FEATURES+ENGINEERED_FEATURES
X=df[list(COMBINED_FEATURES)].copy(); family=df['attack_category'].astype(str); binary=(family!='Normal').astype(int)
train_idx,test_idx=train_test_split(np.arange(len(df)),test_size=0.20,stratify=family,random_state=RANDOM_STATE)
normal_train_idx=train_idx[family.iloc[train_idx].to_numpy()=='Normal']
if len(normal_train_idx)>NORMAL_TRAIN_LIMIT: normal_train_idx,_=train_test_split(normal_train_idx,train_size=NORMAL_TRAIN_LIMIT,random_state=RANDOM_STATE)
X_normal=X.iloc[normal_train_idx].copy(); X_test=X.iloc[test_idx].copy(); y_test=binary.iloc[test_idx].to_numpy()
categorical_features=X_normal.select_dtypes(include=['object','category','string']).columns.tolist(); numeric_features=[f for f in COMBINED_FEATURES if f not in categorical_features]
print(f'Combined raw features={len(COMBINED_FEATURES)} ({len(ORIGINAL_FEATURES)} original + {len(ENGINEERED_FEATURES)} engineered); normal train={len(X_normal)}; test={len(X_test)}; attacks={int(y_test.sum())}')

Combined raw features=72 (41 original + 31 engineered); normal train=6000; test=25195; attacks=11726


In [3]:
selector_preprocessor=ColumnTransformer([('numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',MinMaxScaler())]),numeric_features),('categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('ordinal',OrdinalEncoder(handle_unknown='use_encoded_value',unknown_value=-1)),('scale',MinMaxScaler())]),categorical_features)],remainder='drop',verbose_feature_names_out=False)
selection_indices=np.random.default_rng(RANDOM_STATE).choice(len(X_normal),size=min(SELECTION_SAMPLE_SIZE,len(X_normal)),replace=False)
selection_values=selector_preprocessor.fit_transform(X_normal.iloc[selection_indices]); selector_names=numeric_features+categorical_features
correlation=pd.DataFrame(selection_values,columns=selector_names).corr().abs(); upper=correlation.where(np.triu(np.ones(correlation.shape),k=1).astype(bool)); correlation_removed=[c for c in upper.columns if (upper[c]>CORRELATION_THRESHOLD).any()]; correlation_features=[f for f in COMBINED_FEATURES if f not in correlation_removed]
variance_selector=VarianceThreshold(threshold=VARIANCE_THRESHOLD).fit(selection_values); variance_features=[f for f,keep in zip(selector_names,variance_selector.get_support()) if keep]
W=kneighbors_graph(selection_values,n_neighbors=LAPLACIAN_NEIGHBORS,mode='distance',include_self=False); positive=W.data[W.data>0]; heat=float(np.median(positive)) if len(positive) else 1.0; W.data=np.exp(-(W.data**2)/(2*heat**2)); W=(W+W.T)*0.5; degree=np.asarray(W.sum(axis=1)).ravel(); L=csgraph.laplacian(W,normed=False)
laplacian_scores=[]
for j in range(selection_values.shape[1]):
    values=selection_values[:,j].astype(float); centered=values-np.dot(degree,values)/max(degree.sum(),1e-12); denominator=float(np.dot(degree,centered**2)); laplacian_scores.append(float(centered@(L@centered))/denominator if denominator>1e-12 else np.inf)
laplacian_order=np.argsort(laplacian_scores); laplacian_features=[selector_names[i] for i in laplacian_order[:min(LAPLACIAN_TOP_K,len(selector_names))]]
def make_preprocessor(features):
    nums=[f for f in features if f in numeric_features]; cats=[f for f in features if f in categorical_features]; transformers=[]
    if nums: transformers.append(('numeric',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),nums))
    if cats: transformers.append(('categorical',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cats))
    return ColumnTransformer(transformers,remainder='drop',verbose_feature_names_out=False)
selection_definitions={'none':list(COMBINED_FEATURES),'correlationfiltering':correlation_features,'variancethreshold':variance_features,'laplacianscore':laplacian_features}
representations={}
for method,features in selection_definitions.items():
    transformer=make_preprocessor(features); train_values=transformer.fit_transform(X_normal[features]); test_values=transformer.transform(X_test[features]); engineered=[f for f in features if f in ENGINEERED_FEATURES]
    representations[method]={'transformer':transformer,'selected_features':features,'train':train_values,'test':test_values,'dimensions':train_values.shape[1],'engineered_count':len(engineered),'engineered_percent':100*len(engineered)/max(len(features),1),'engineered_features':engineered}
full_transformer=representations['none']['transformer']; full_train=representations['none']['train']; full_test=representations['none']['test']; pca=PCA(n_components=PCA_VARIANCE_TO_KEEP,svd_solver='full',random_state=RANDOM_STATE).fit(full_train); pca_train=pca.transform(full_train); pca_test=pca.transform(full_test)
encoded_names=list(full_transformer.get_feature_names_out()); encoded_importance=(np.abs(pca.components_)*pca.explained_variance_ratio_[:,None]).sum(axis=0); raw_contribution={f:float(sum(encoded_importance[i] for i,n in enumerate(encoded_names) if n==f or n.startswith(f+'_'))) for f in COMBINED_FEATURES}; pca_engineered_percent=100*sum(raw_contribution[f] for f in ENGINEERED_FEATURES)/max(sum(raw_contribution.values()),1e-12)
pca_transformer=Pipeline([('preprocessor',full_transformer),('pca',pca)]); representations['pca']={'transformer':pca_transformer,'selected_features':[f'PC{i+1}' for i in range(pca_train.shape[1])],'train':pca_train,'test':pca_test,'dimensions':pca_train.shape[1],'engineered_count':np.nan,'engineered_percent':pca_engineered_percent,'engineered_features':[],'raw_contribution':raw_contribution}
selection_details={'none':{'method':'none'},'correlationfiltering':{'threshold':CORRELATION_THRESHOLD,'removed_features':correlation_removed},'variancethreshold':{'threshold':VARIANCE_THRESHOLD,'variances':{f:float(v) for f,v in zip(selector_names,variance_selector.variances_)}},'laplacianscore':{'top_k':LAPLACIAN_TOP_K,'neighbors':LAPLACIAN_NEIGHBORS,'scores':{f:(None if not np.isfinite(v) else float(v)) for f,v in zip(selector_names,laplacian_scores)}},'pca':{'variance_to_keep':PCA_VARIANCE_TO_KEEP,'explained_variance':float(pca.explained_variance_ratio_.sum()),'raw_feature_contribution':raw_contribution}}
representation_summary_df=pd.DataFrame([{'feature_selection':m,'raw_selected_count':(len(r['selected_features']) if m!='pca' else np.nan),'encoded_dimensions':r['dimensions'],'dimension_reduction_percent':100*(1-r['dimensions']/representations['none']['dimensions']),'engineered_selected_count':r['engineered_count'],'engineered_percent_of_selection_or_contribution':r['engineered_percent']} for m,r in representations.items()]).sort_values('encoded_dimensions')
display(representation_summary_df)

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\joblib\externals\loky\backend\context.py:131: UserWarning: Could not find the number of physical cores for the following reason:
found 0 physical cores < 1
Returning the number of logical cores instead. You can silence this warning by setting LOKY_MAX_CPU_COUNT to the number of cores you want to use.
  warnings.warn(
  File "D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\joblib\externals\loky\backend\context.py", line 255, in _count_physical_cores
    raise ValueError(f"found {cpu_count_physical} physical cores < 1")


,feature_selection,raw_selected_count,encoded_dimensions,dimension_reduction_percent,engineered_selected_count,engineered_percent_of_selection_or_contribution
4,pca,NaN,31,69.902913,NaN,45.181642
2,variancethreshold,34.0,65,36.893204,15.0,44.117647
3,laplacianscore,36.0,67,34.951456,16.0,44.444444
1,correlationfiltering,56.0,80,22.330097,19.0,33.928571
0,none,72.0,103,0.000000,31.0,43.055556


In [4]:
DEFAULT_PARAMS={'IsolationForest':{'n_estimators':300,'max_samples':1.0,'contamination':'auto','n_jobs':-1},'LocalOutlierFactor':{'n_neighbors':35,'contamination':0.05,'novelty':True,'n_jobs':-1},'OneClassSVM':{'kernel':'rbf','nu':0.05,'gamma':'scale'}}
def build_detector(name):
    if name=='IsolationForest': return IsolationForest(random_state=RANDOM_STATE,**DEFAULT_PARAMS[name])
    if name=='LocalOutlierFactor': return LocalOutlierFactor(**DEFAULT_PARAMS[name])
    return OneClassSVM(**DEFAULT_PARAMS[name])
def anomaly_score(model,values): return -np.asarray(model.decision_function(values)).ravel()
def metrics_for(y,pred,scores):
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel(); return {'accuracy':accuracy_score(y,pred),'balanced_accuracy':balanced_accuracy_score(y,pred),'precision':precision_score(y,pred,zero_division=0),'recall':recall_score(y,pred,zero_division=0),'f1':f1_score(y,pred,zero_division=0),'roc_auc':roc_auc_score(y,scores),'average_precision':average_precision_score(y,scores),'false_positive_rate':fp/max(fp+tn,1),'detection_rate':tp/max(tp+fn,1),'specificity':tn/max(tn+fp,1),'true_negatives':int(tn),'false_positives':int(fp),'false_negatives':int(fn),'true_positives':int(tp)}
def diagnostic_figure(y,pred,scores,threshold,title):
    fig,axes=plt.subplots(2,2,figsize=(11,8)); ConfusionMatrixDisplay.from_predictions(y,pred,display_labels=['Normal','Attack'],cmap='Blues',colorbar=False,ax=axes[0,0]); RocCurveDisplay.from_predictions(y,scores,ax=axes[0,1]); PrecisionRecallDisplay.from_predictions(y,scores,ax=axes[1,0]); axes[1,1].hist(scores[y==0],bins=40,alpha=.6,label='Normal'); axes[1,1].hist(scores[y==1],bins=40,alpha=.6,label='Attack'); axes[1,1].axvline(threshold,color='black',ls='--'); axes[1,1].legend(); axes[1,1].set_title('Anomaly-score distribution'); fig.suptitle(title); fig.tight_layout(); return fig
class RawAnomalyModel(BaseEstimator):
    def __init__(self,transformer,detector,threshold,required_features): self.transformer=transformer; self.detector=detector; self.threshold=threshold; self.required_features=required_features
    def decision_function(self,frame): return -np.asarray(self.detector.decision_function(self.transformer.transform(frame[self.required_features]))).ravel()
    def predict(self,frame): return (self.decision_function(frame)>=self.threshold).astype(int)
train_tracking=X_normal.copy(); train_tracking['binary_target']='Normal'; test_tracking=X_test.copy(); test_tracking['binary_target']=np.where(y_test==1,'Attack','Normal')
train_dataset=mlflow.data.from_pandas(train_tracking,source=str(data_path.resolve()),targets='binary_target',name='combined_normal_only_feature_selection_train'); test_dataset=mlflow.data.from_pandas(test_tracking,source=str(data_path.resolve()),targets='binary_target',name='combined_feature_selection_anomaly_test')
common_metadata={'original_features':list(ORIGINAL_FEATURES),'engineered_features':list(ENGINEERED_FEATURES),'combined_features':list(COMBINED_FEATURES),'normal_only_training':True,'labels_used_for_feature_selection':False,'python':platform.python_version(),'sklearn':sklearn.__version__,'input_schema':{c:str(X[c].dtype) for c in COMBINED_FEATURES}}

D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(
D:\E Drive\Sentiflow-Network Intrusion Detection\.venv\lib\site-packages\mlflow\data\dataset_source_registry.py:148: UserWarning: The specified dataset source can be interpreted in multiple ways: LocalArtifactDatasetSource, LocalArtifactDatasetSource. MLflow will assume that this is a LocalArtifactDatasetSource source.
  return _dataset_source_registry.resolve(


## Execute and track all 15 combinations

The threshold for every run is the 95th percentile of that model's anomaly score on normal-only training data. Larger model scores mean more anomalous traffic.

In [5]:
results=[]; fitted_models={}
for feature_selection,representation in representations.items():
    for algorithm in ALGORITHMS:
        run_name=f'combined_{feature_selection}_{algorithm}'; started=time.perf_counter(); detector=build_detector(algorithm); detector.fit(representation['train']); fit_seconds=time.perf_counter()-started; train_scores=anomaly_score(detector,representation['train']); threshold=float(np.quantile(train_scores,0.95)); scores=anomaly_score(detector,representation['test']); prediction=(scores>=threshold).astype(int); metrics=metrics_for(y_test,prediction,scores); metrics['fit_seconds']=fit_seconds
        required_features=list(COMBINED_FEATURES) if feature_selection=='pca' else list(representation['selected_features']); raw_model=RawAnomalyModel(representation['transformer'],detector,threshold,required_features); input_example=X_test[list(COMBINED_FEATURES)].head(5); sample_scores=raw_model.decision_function(input_example); sample_output=pd.DataFrame({'binary_prediction':raw_model.predict(input_example),'anomaly_score':sample_scores})
        with mlflow.start_run(run_name=run_name) as run:
            mlflow.set_tags({'task':'feature_selection_anomaly_detection','data_variant':'Combined','feature_selection':feature_selection,'algorithm':algorithm,'training_data':'normal_only','labels_used_for_selection':'false'})
            mlflow.log_input(train_dataset,context='normal_only_training'); mlflow.log_input(test_dataset,context='held_out_evaluation')
            mlflow.log_params({**DEFAULT_PARAMS[algorithm],'algorithm':algorithm,'feature_selection':feature_selection,'threshold':threshold,'threshold_quantile':0.95,'input_raw_features':len(COMBINED_FEATURES),'selected_raw_features':(len(representation['selected_features']) if feature_selection!='pca' else 'components_not_raw_features'),'representation_dimensions':representation['dimensions'],'dimension_reduction_percent':100*(1-representation['dimensions']/representations['none']['dimensions']),'engineered_selected_count':representation['engineered_count'],'engineered_selection_or_contribution_percent':representation['engineered_percent'],'normal_train_rows':len(X_normal),'test_rows':len(X_test),'selection_sample_rows':len(selection_indices),'random_state':RANDOM_STATE})
            mlflow.log_metrics({k:float(v) for k,v in metrics.items()})
            mlflow.log_dict({'selected_features_or_components':representation['selected_features'],'selected_engineered_features':representation['engineered_features'],'selection_details':selection_details[feature_selection]},'feature_selection/selected_features_and_details.json'); mlflow.log_dict(common_metadata,'metadata/run_metadata.json'); mlflow.log_dict({'input':{'type':'pandas.DataFrame','required_combined_columns':list(COMBINED_FEATURES)},'output':{'binary_prediction':{'0':'Normal','1':'Attack'},'anomaly_score':'larger means more anomalous'},'decision_rule':'anomaly_score >= threshold'},'metadata/input_output_contract.json'); mlflow.log_table(input_example.reset_index(drop=True),'examples/input_example.json'); mlflow.log_table(sample_output,'examples/output_example.json')
            fig=diagnostic_figure(y_test,prediction,scores,threshold,run_name); mlflow.log_figure(fig,'plots/anomaly_diagnostics.png'); plt.close(fig); signature=infer_signature(input_example,raw_model.predict(input_example)); mlflow.sklearn.log_model(sk_model=raw_model,name='raw_input_anomaly_model',signature=signature,input_example=input_example,serialization_format=mlflow.sklearn.SERIALIZATION_FORMAT_CLOUDPICKLE)
            row={'run_name':run_name,'run_id':run.info.run_id,'feature_selection':feature_selection,'algorithm':algorithm,'raw_selected_count':(len(representation['selected_features']) if feature_selection!='pca' else np.nan),'dimensions':representation['dimensions'],'reduction_percent':100*(1-representation['dimensions']/representations['none']['dimensions']),'engineered_percent':representation['engineered_percent'],**metrics}; results.append(row); fitted_models[(feature_selection,algorithm)]=raw_model
        print(f"{run_name}: dims={representation['dimensions']}, F1={metrics['f1']:.4f}, DR={metrics['detection_rate']:.4f}, FPR={metrics['false_positive_rate']:.4f}")
results_df=pd.DataFrame(results).sort_values('f1',ascending=False).reset_index(drop=True); display(results_df[['run_name','dimensions','reduction_percent','engineered_percent','accuracy','balanced_accuracy','precision','detection_rate','false_positive_rate','f1','roc_auc','average_precision']])

Completed and logged 15 combinations (5 representations × 3 anomaly algorithms). The comparison tables below were reconstructed from the verified FINISHED MLflow runs.


In [6]:
if 'results_df' not in globals():
    experiment=mlflow.get_experiment_by_name(EXPERIMENT_NAME); tracked=mlflow.search_runs([experiment.experiment_id],filter_string="tags.task = 'feature_selection_anomaly_detection'").sort_values('start_time',ascending=False).drop_duplicates('tags.mlflow.runName')
    results_df=pd.DataFrame({'run_name':tracked['tags.mlflow.runName'],'run_id':tracked['run_id'],'feature_selection':tracked['tags.feature_selection'],'algorithm':tracked['tags.algorithm'],'raw_selected_count':pd.to_numeric(tracked['params.selected_raw_features'],errors='coerce'),'dimensions':pd.to_numeric(tracked['params.representation_dimensions'],errors='coerce'),'reduction_percent':pd.to_numeric(tracked['params.dimension_reduction_percent'],errors='coerce'),'engineered_percent':pd.to_numeric(tracked['params.engineered_selection_or_contribution_percent'],errors='coerce'),'accuracy':tracked['metrics.accuracy'],'balanced_accuracy':tracked['metrics.balanced_accuracy'],'precision':tracked['metrics.precision'],'recall':tracked['metrics.recall'],'f1':tracked['metrics.f1'],'roc_auc':tracked['metrics.roc_auc'],'average_precision':tracked['metrics.average_precision'],'false_positive_rate':tracked['metrics.false_positive_rate'],'detection_rate':tracked['metrics.detection_rate'],'specificity':tracked['metrics.specificity']}).sort_values('f1',ascending=False).reset_index(drop=True)
baseline=results_df[results_df.feature_selection=='none'].set_index('algorithm'); comparison_df=results_df.copy(); comparison_df['f1_delta_vs_combined_none']=comparison_df.apply(lambda r:r.f1-baseline.loc[r.algorithm,'f1'],axis=1); comparison_df['detection_rate_delta']=comparison_df.apply(lambda r:r.detection_rate-baseline.loc[r.algorithm,'detection_rate'],axis=1); comparison_df['fpr_delta']=comparison_df.apply(lambda r:r.false_positive_rate-baseline.loc[r.algorithm,'false_positive_rate'],axis=1)
best_by_algorithm=comparison_df.sort_values(['f1','balanced_accuracy'],ascending=False).groupby('algorithm',as_index=False).first()[['algorithm','feature_selection','dimensions','reduction_percent','engineered_percent','detection_rate','false_positive_rate','f1','f1_delta_vs_combined_none']]
selected_only=comparison_df[comparison_df.feature_selection!='none']; best_selected=selected_only.sort_values(['f1','balanced_accuracy'],ascending=False).iloc[0]; its_baseline=baseline.loc[best_selected.algorithm]
raw_methods=['correlationfiltering','variancethreshold','laplacianscore']; frequency=pd.Series({f:sum(f in representations[m]['selected_features'] for m in raw_methods) for f in COMBINED_FEATURES},name='selection_frequency').sort_values(ascending=False); consistency_df=frequency.reset_index().rename(columns={'index':'feature'}); consistency_df['feature_type']=np.where(consistency_df.feature.isin(ENGINEERED_FEATURES),'Engineered','Original')
engineered_retention_df=representation_summary_df[['feature_selection','raw_selected_count','encoded_dimensions','dimension_reduction_percent','engineered_selected_count','engineered_percent_of_selection_or_contribution']].copy()
display(Markdown('## Performance change relative to the full combined baseline')); display(comparison_df[['algorithm','feature_selection','dimensions','detection_rate','false_positive_rate','f1','f1_delta_vs_combined_none','detection_rate_delta','fpr_delta']].sort_values(['algorithm','f1'],ascending=[True,False])); display(Markdown('## Best representation for each detector')); display(best_by_algorithm); display(Markdown('## Engineered-feature retention')); display(engineered_retention_df); display(Markdown('## Features selected consistently by raw selectors')); display(consistency_df.head(30))
fig,axes=plt.subplots(1,2,figsize=(13,4)); pivot=comparison_df.pivot(index='feature_selection',columns='algorithm',values='f1'); pivot.plot(kind='bar',ax=axes[0]); axes[0].set_ylabel('Test F1'); axes[0].set_title('Anomaly F1 by representation'); axes[0].tick_params(axis='x',rotation=30); representation_summary_df.plot(x='feature_selection',y='engineered_percent_of_selection_or_contribution',kind='bar',legend=False,ax=axes[1],color='darkorange'); axes[1].set_ylabel('Engineered feature %'); axes[1].set_title('Engineered retention / PCA contribution'); axes[1].tick_params(axis='x',rotation=30); fig.tight_layout(); display(fig)
hybrid_recommended=(best_selected.f1>=its_baseline.f1-0.005) and (best_selected.reduction_percent>=20); improvement_count=int((selected_only.f1_delta_vs_combined_none>0).sum()); total_selected=len(selected_only); consistent_engineered=consistency_df[(consistency_df.feature_type=='Engineered')&(consistency_df.selection_frequency==len(raw_methods))].feature.tolist(); consistent_original=consistency_df[(consistency_df.feature_type=='Original')&(consistency_df.selection_frequency==len(raw_methods))].feature.tolist()
recommendation=('Yes. The selected representation preserves or improves F1 while reducing at least 20% of encoded dimensions, so it is a strong input candidate for a hybrid supervised/anomaly pipeline.' if hybrid_recommended else 'Not yet as a universal replacement. Use the selected representation as an additional hybrid branch or validate it with supervised cross-validation, because its anomaly F1 gain or dimensionality benefit is not consistently strong enough.')
display(Markdown(f'''# Answers and inferences

## Has feature selection improved anomaly-detection performance?
Feature selection improves test F1 in **{improvement_count} of {total_selected}** feature-selected detector combinations. The strongest selected combination is **{best_selected.feature_selection} + {best_selected.algorithm}** with F1 **{best_selected.f1:.4f}**, detection rate **{best_selected.detection_rate:.4f}**, and FPR **{best_selected.false_positive_rate:.4f}**. Against the same algorithm's full combined baseline, its F1 changes by **{best_selected.f1_delta_vs_combined_none:+.4f}** while reducing encoded dimensions by **{best_selected.reduction_percent:.1f}%**.

## Which selector works best for each algorithm?
The `best_by_algorithm` table gives the measured answer and includes the no-selection baseline when no selector wins. This is important because a representation can raise detection rate simply by increasing false alarms; F1 and FPR must be considered together.

## Engineered-feature insights
The combined input starts with **31 engineered of 72 raw features (43.1%)**. For raw selectors, `engineered_percent_of_selection_or_contribution` is the percentage of retained raw columns that are engineered. For PCA it is the explained-variance-weighted loading contribution from engineered inputs, not a selected-column percentage. Engineered features retained by all three raw selectors are: **{', '.join(consistent_engineered) if consistent_engineered else 'none'}**. Original features retained by all three are: **{', '.join(consistent_original)}**.

## Should selected features be used for hybrid modelling?
**{recommendation}** A practical hybrid design should combine the best anomaly score with a supervised classifier trained on the selected raw features. Prefer an interpretable raw selector over PCA when attack explanations and feature-level monitoring are required; prefer PCA only when its compression benefit is decisive and interpretability is secondary.

## Caution
These results measure global Normal-vs-Attack detection. Before deployment, repeat attack-family and time-based validation because a feature set that helps common DoS traffic may still suppress rare R2L or U2R signals.''' ))
with mlflow.start_run(run_id=str(best_selected.run_id)):
    mlflow.log_table(comparison_df,'comparison/all_feature_selection_results.json'); mlflow.log_table(best_by_algorithm,'comparison/best_representation_per_algorithm.json'); mlflow.log_table(consistency_df,'feature_selection/selection_consistency.json'); mlflow.log_figure(fig,'comparison/summary.png')
plt.close(fig)

## Performance change relative to the full combined baseline

,algorithm,feature_selection,dimensions,detection_rate,false_positive_rate,f1,f1_delta_vs_combined_none,detection_rate_delta,fpr_delta
0,IsolationForest,correlationfiltering,80,0.959236,0.050635,0.950964,0.005670,0.010234,-0.000594
1,IsolationForest,none,103,0.949002,0.051229,0.945294,0.000000,0.000000,0.000000
2,IsolationForest,laplacianscore,67,0.950026,0.052714,0.945029,-0.000265,0.001023,0.001485
3,IsolationForest,variancethreshold,65,0.946444,0.050783,0.944189,-0.001105,-0.002558,-0.000445
8,IsolationForest,pca,31,0.893570,0.051674,0.915109,-0.030185,-0.055432,0.000445
10,LocalOutlierFactor,pca,31,0.584087,0.056649,0.708346,0.010283,0.013986,0.001559
11,LocalOutlierFactor,correlationfiltering,80,0.577264,0.054347,0.704114,0.006051,0.007164,-0.000742
12,LocalOutlierFactor,none,103,0.570101,0.055089,0.698063,0.000000,0.000000,0.000000
13,LocalOutlierFactor,variancethreshold,65,0.384274,0.050561,0.532845,-0.165218,-0.185826,-0.004529
14,LocalOutlierFactor,laplacianscore,67,0.261129,0.050709,0.395837,-0.302226,-0.308972,-0.004380


## Best representation for each detector

,algorithm,feature_selection,dimensions,reduction_percent,engineered_percent,detection_rate,false_positive_rate,f1,f1_delta_vs_combined_none
0,IsolationForest,correlationfiltering,80,22.330097,33.928571,0.959236,0.050635,0.950964,0.005670
1,LocalOutlierFactor,pca,31,69.902913,45.181642,0.584087,0.056649,0.708346,0.010283
2,OneClassSVM,variancethreshold,65,36.893204,44.117647,0.912417,0.055758,0.923283,0.002981


## Engineered-feature retention

,feature_selection,raw_selected_count,encoded_dimensions,dimension_reduction_percent,engineered_selected_count,engineered_percent_of_selection_or_contribution
4,pca,NaN,31,69.902913,NaN,45.181642
2,variancethreshold,34.0,65,36.893204,15.0,44.117647
3,laplacianscore,36.0,67,34.951456,16.0,44.444444
1,correlationfiltering,56.0,80,22.330097,19.0,33.928571
0,none,72.0,103,0.000000,31.0,43.055556


## Features selected consistently by raw selectors

,feature,selection_frequency,feature_type
0,protocoltype,3,Original
1,service,3,Original
2,diffsrvrate,3,Original
3,samesrvrate,3,Original
4,dsthostsamesrcportrate,3,Original
5,dsthostdiffsrvrate,3,Original
6,isguestlogin,3,Original
7,loggedin,3,Original
8,serror_rate_gap,3,Engineered
9,different_host_service_connections,3,Engineered


<Figure size 1300x400 with 2 Axes>

# Answers and inferences

## Has feature selection improved anomaly-detection performance?
Feature selection improves test F1 in **5 of 12** feature-selected detector combinations. The strongest selected combination is **correlationfiltering + IsolationForest** with F1 **0.9510**, detection rate **0.9592**, and FPR **0.0506**. Against the same algorithm's full combined baseline, its F1 changes by **+0.0057** while reducing encoded dimensions by **22.3%**.

## Which selector works best for each algorithm?
The `best_by_algorithm` table gives the measured answer and includes the no-selection baseline when no selector wins. This is important because a representation can raise detection rate simply by increasing false alarms; F1 and FPR must be considered together.

## Engineered-feature insights
The combined input starts with **31 engineered of 72 raw features (43.1%)**. For raw selectors, `engineered_percent_of_selection_or_contribution` is the percentage of retained raw columns that are engineered. For PCA it is the explained-variance-weighted loading contribution from engineered inputs, not a selected-column percentage. Engineered features retained by all three raw selectors are: **serror_rate_gap, different_host_service_connections, same_source_port_pressure, different_service_rate_gap, host_service_ratio, dst_byte_fraction, host_scan_pressure, src_byte_fraction**. Original features retained by all three are: **protocoltype, service, diffsrvrate, samesrvrate, dsthostsamesrcportrate, dsthostdiffsrvrate, isguestlogin, loggedin, dsthostcount, dsthostrerrorrate, dsthostsrvcount, dsthostsamesrvrate, rerrorrate, dsthostsrvrerrorrate**.

## Should selected features be used for hybrid modelling?
**Yes. The selected representation preserves or improves F1 while reducing at least 20% of encoded dimensions, so it is a strong input candidate for a hybrid supervised/anomaly pipeline.** A practical hybrid design should combine the best anomaly score with a supervised classifier trained on the selected raw features. Prefer an interpretable raw selector over PCA when attack explanations and feature-level monitoring are required; prefer PCA only when its compression benefit is decisive and interpretability is secondary.

## Caution
These results measure global Normal-vs-Attack detection. Before deployment, repeat attack-family and time-based validation because a feature set that helps common DoS traffic may still suppress rare R2L or U2R signals.